# QLoRA Fine-Tuning for Multimodal Alignment of Voxtral with GLaDOS Persona

In [1]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoProcessor,
    VoxtralForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import os
import gc
import wandb
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="bitsandbytes")
HYPERTUNE = False # Set to True to run hyperparameter sweep, False for single training run

wandb.login()
os.environ["WANDB_PROJECT"] = "Voxtral-GLaDOS-Multimodal"
os.environ["WANDB_LOG_MODEL"] = "false"
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "mistralai/Voxtral-Mini-3B-2507"
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "left"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # Highly optimized for speed/accuracy
    bnb_4bit_use_double_quant=True, # Saves extra memory at no speed cost
    bnb_4bit_compute_dtype=compute_dtype
)

#### Dataset Formatting for Multimodal SFT

In [3]:
df = pd.read_csv("./data/combined_multimodal_dataset_train.csv")
# Drop rows missing critical components
df = df.dropna(subset=['Assistant_Payload', 'Target_GLaDOS_Response', 'Audio_File'])
# Extract all unique user commands
unique_commands = df['User_Command'].unique()
train_commands, eval_commands = train_test_split(unique_commands, test_size=0.2, random_state=42)
# Filter the dataframe so all variations of a command stay strictly together
df_train = df[df['User_Command'].isin(train_commands)]
df_eval = df[df['User_Command'].isin(eval_commands)]
del df
print(f"Total Unique Commands: {len(unique_commands)}")
print(f"Train rows: {len(df_train)} | Eval rows: {len(df_eval)}")

def format_dataframe_to_dataset(df, audio_dir="./data/synthesized_train/"):
    dataset_dict = {"messages": []}
    missing_files = 0
    for _, row in df.iterrows():
        full_audio_path = os.path.join(audio_dir, row["Audio_File"])
        # Validation check to prevent the trainer from crashing mid-epoch
        if not os.path.exists(full_audio_path):
            missing_files += 1
            print(f"Warning: Skipping {row['Audio_File']} - File not found at {full_audio_path}")
            continue
        target_output = f"{row['Assistant_Payload']}\n\n{row['Target_GLaDOS_Response']}{processor.tokenizer.eos_token}"
        conversation = [
            {"role": "user", "content": [{"type": "audio", "path": full_audio_path}]}, # The processor will handle loading and feature extraction during training
            {"role": "assistant", "content": [{"type": "text", "text": target_output}]},
            {"role": "user", "content": [{"type": "text", "text": "DUMMY_STOP"}]} # A dummy user turn to satisfy the Mistral Serving Validator
        ]
        dataset_dict["messages"].append(conversation)
    if missing_files > 0:
        print(f"Warning: Skipped {missing_files} rows due to missing audio files.")
    # No need to cast to datasets.Audio(), saving massive amounts of RAM
    return Dataset.from_dict(dataset_dict)

print("\nFormatting Train Dataset...")
train_dataset = format_dataframe_to_dataset(df_train)
print("Formatting Eval Dataset...")
eval_dataset = format_dataframe_to_dataset(df_eval)

Formatting multimodal dataset...


In [ ]:
def voxtral_collate_fn(batch):
    conversations = [item["messages"] for item in batch]
    # Let the processor handle the audio loading, padding, and tokenization.
    inputs = processor.apply_chat_template(
        conversations,
        tokenize=True,
        return_dict=True,
        processor_kwargs={
            "padding": True,
            "return_tensors": "pt",
        }
    )
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    # 1. Slice off the dummy turn
    for i in range(input_ids.shape[0]):
        # Find all occurrences of the EOS token in this sequence
        eos_indices = (input_ids[i] == processor.tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
        if len(eos_indices) > 0:
            # The first EOS belongs to our Assistant response
            target_eos_idx = eos_indices[0]
            # Overwrite everything after the EOS token with padding
            input_ids[i, target_eos_idx + 1:] = processor.tokenizer.pad_token_id
            attention_mask[i, target_eos_idx + 1:] = 0
    # 2. Mask user audio and padding tokens
    labels = input_ids.clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    # 3. Mask User Input (The Audio & Prompt) so loss is ONLY calculated on the Assistant's JSON/Text
    # Mistral/Voxtral uses the [/INST] tag to separate user prompt from assistant response
    inst_token_id = processor.tokenizer.convert_tokens_to_ids("[/INST]")
    for i in range(labels.shape[0]):
        try:
            # Find the last occurrence of [/INST] (end of user instruction)
            inst_indices = (labels[i] == inst_token_id).nonzero(as_tuple=True)[0]
            if len(inst_indices) > 0:
                # We mask up to the FIRST [/INST], which contains the real user audio prompt
                # This ensures the model is ONLY graded on the Assistant's text
                first_inst_idx = inst_indices[0]
                labels[i, :first_inst_idx + 1] = -100
        except IndexError:
            pass # Fallback in case of formatting anomaly
    inputs["input_ids"] = input_ids
    inputs["attention_mask"] = attention_mask
    inputs["labels"] = labels
    return inputs

#### Hyperpatameters tuning

In [ ]:
def create_sweep_func(train_dataset, eval_dataset):
    def train_sweep():
        wandb.init()
        config = wandb.config
        # Reload Base Model (clears old adapters from memory)
        model = VoxtralForConditionalGeneration.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            attn_implementation="flash_attention_2",
            device_map=device
        )
        model = prepare_model_for_kbit_training(model)
        model.get_input_embeddings().to(compute_dtype)
        if hasattr(model, "get_output_embeddings") and model.get_output_embeddings() is not None:
            model.get_output_embeddings().to(compute_dtype)
        if hasattr(model, "audio_encoder"):
            model.audio_encoder.to(compute_dtype)
        model.config.pad_token_id = processor.tokenizer.pad_token_id
        model.config.eos_token_id = processor.tokenizer.eos_token_id
        model.config.bos_token_id = processor.tokenizer.bos_token_id

        # Dynamic LoRA Config from Sweep
        lora_config = LoraConfig(
            r=config.lora_r,
            lora_alpha=config.lora_alpha,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_dropout=config.lora_dropout,
            bias="none",
            task_type="CAUSAL_LM"
        )
        # Dynamic Training Args
        training_args = SFTConfig(
            output_dir="./models/voxtral-sweep",
            per_device_train_batch_size=2,
            per_device_eval_batch_size=2,
            eval_strategy="steps",
            eval_steps=100,
            save_strategy="no",
            load_best_model_at_end=False,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            gradient_accumulation_steps=4,
            dataloader_num_workers=4,
            dataloader_pin_memory=True,
            learning_rate=config.learning_rate,
            max_steps=100, # KEEP THIS LOW FOR SWEEPS (e.g., 100 steps to see if loss drops)
            logging_steps=10,
            optim="paged_adamw_8bit",
            bf16=torch.cuda.is_bf16_supported(),
            fp16=not torch.cuda.is_bf16_supported(),
            remove_unused_columns=False,
            dataset_kwargs={"skip_prepare_dataset": True},
            report_to="wandb"
        )
        # Initialize Trainer
        trainer = SFTTrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset, # Assuming train_dataset is already loaded in memory
            eval_dataset=eval_dataset,
            data_collator=voxtral_collate_fn,
            peft_config=lora_config,
        )

        trainer.train()
    return train_sweep

if HYPERTUNE:
    project = "Voxtral-GLaDOS-Multimodal"
    sweep_config = {
        'method': 'bayes', # Bayesian optimization (smarter than random search)
        'metric': {
            'name': 'train/loss',
            'goal': 'minimize'
        },
        'parameters': {
            'learning_rate': {
                'distribution': 'log_uniform_values',
                'min': 1e-5,
                'max': 5e-4
            },
            'lora_r': {
                'values': [8, 16, 32] # Rank of the adapters
            },
            'lora_alpha': {
                'values': [16, 32, 64] # Scaling factor
            },
            'lora_dropout': {
                'values': [0.05, 0.1]
            }
        }
    }
    sweep_id = wandb.sweep(sweep_config, project=project)
    # Execute the sweep (Run 5 different combinations)
    wandb.agent(sweep_id, create_sweep_func(train_dataset, eval_dataset), count=5)
    print("\nSweep complete! Fetching the best parameters from W&B cloud...")
    api = wandb.Api()
    entity = api.default_entity
    sweep = api.sweep(f"{entity}/{project}/{sweep_id}")
    best_run = sweep.best_run()
    best_params = best_run.config
    print(f"Best Parameters Found: {best_params}")
else:
    # If HYPERTUNE is False, fall back to your safe manual defaults
    best_params = {
        'learning_rate': 2e-4,
        'lora_r': 16,
        'lora_alpha': 32,
        'lora_dropout': 0.05
    }
    print(f"Skipping sweep. Using default parameters: {best_params}")

#### Supervised Fine-Tuning with TRL's SFTTrainer

In [4]:
print(f"Loading {model_id} in 4-bit precision...")
model = VoxtralForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    attn_implementation="flash_attention_2",
    device_map=device
)

# Prepare model for gradient training
model = prepare_model_for_kbit_training(model)
# Revert the text embeddings back to bfloat16/float16 to match the Audio Encoder
model.get_input_embeddings().to(compute_dtype)
# Also ensure the output layer matches
if hasattr(model, "get_output_embeddings") and model.get_output_embeddings() is not None:
    model.get_output_embeddings().to(compute_dtype)
# It is also good practice to ensure the audio encoder didn't get accidentally cast to float32
if hasattr(model, "audio_encoder"):
    model.audio_encoder.to(compute_dtype)
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id
model.config.bos_token_id = processor.tokenizer.bos_token_id

lora_config = LoraConfig(
    r=best_params['lora_r'],
    lora_alpha=best_params['lora_alpha'],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=best_params['lora_dropout'],
    bias="none",
    task_type="CAUSAL_LM"
)

training_args = SFTConfig(
    output_dir="./models/voxtral-glados-sft",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    gradient_accumulation_steps=4,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    learning_rate=best_params['learning_rate'],
    logging_steps=10,
    max_steps=500, # Adjust based on dataset size
    optim="adamw_bnb_8bit", # paged_adamw_8bit use ram if vram is saturated (paging)
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    remove_unused_columns=False, # Crucial so the collator receives the dicts
    dataset_kwargs={"skip_prepare_dataset": True},
    report_to="wandb",
    run_name="voxtral-3b-run1"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=voxtral_collate_fn,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()

trainable params: 30,965,760 || all params: 4,707,236,864 || trainable%: 0.6578


In [ ]:
print("Initiating QLoRA Multimodal Alignment...")
trainer.train()

# Save the final adapter weights
trainer.model.save_pretrained("./models/voxtral-glados-final-adapters")
processor.save_pretrained("./models/voxtral-glados-final-adapters")
print("Training complete. Adapters saved.")

Initiating QLoRA Multimodal Alignment...


/home/vito/miniconda3/envs/pyqwen/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/vito/miniconda3/envs/pyqwen/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
